<a href="https://colab.research.google.com/github/Vunhattan0210/clinical_computational_genomics/blob/main/WGS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Kết nối với drive google

In [17]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Di chuyển tới thư mục


In [18]:
%cd /content/drive/MyDrive/WGS_demo
!pwd
!ls
!ls raw
!ls ref

/content/drive/MyDrive/WGS_demo
/content/drive/MyDrive/WGS_demo
raw  ref
'Bản sao của SRR33305440_1.fastq.gz'   SRR33305440_1.fastq.gz
s_aureus.fna


Cài đặt các tool cần thiết

In [ ]:
%%bash
apt-get update -qq
apt-get install -y -qq fastqc fastp bwa samtools default-jre
pip install -q multiqc

Tạo thư mục con

In [ ]:
%%bash
mkdir -p qc/raw_fastqc qc/trim_fastqc qc
pwd
ls
ls raw
ls ref

Kiểm tra chất lượng dữ liệu

In [ ]:
%%bash
fastqc -t 4 -o qc/raw_fastqc \
  raw/SRR33305440_1.fastq.gz \
  raw/SRR33305440_2.fastq.gz

ls qc/raw_fastqc

Tải file kiểm tra chất lượng dữ liệu về local

In [ ]:
from google.colab import files

files.download("qc/raw_fastqc/SRR33305440_1_fastqc.html")
files.download("qc/raw_fastqc/SRR33305440_2_fastqc.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Tiền xử lý dữ liệu

In [ ]:
%%bash
fastp \
  -i raw/SRR33305440_1.fastq.gz \
  -I raw/SRR33305440_2.fastq.gz \
  -o out/SRR33305440.trim.R1.fastq.gz \
  -O out/SRR33305440.trim.R2.fastq.gz \
  -h qc/SRR33305440.fastp.html \
  -j qc/SRR33305440.fastp.json

Tải kết quả tiền xử lý dữ liệu về

In [ ]:
from google.colab import files
files.download("qc/SRR33305440.fastp.html")

Xem trực tiếp kết quả mà không cần tải về

In [ ]:
from IPython.display import HTML, display
display(HTML(open("qc/SRR33305440.fastp.html").read()))

Kiểm tra chất lượng dữ liệu sau tiền xử lý

In [ ]:
%%bash
fastqc -t 2 -o qc/trim_fastqc \
  out/SRR33305440.trim.R1.fastq.gz \
  out/SRR33305440.trim.R2.fastq.gz
ls qc/trim_fastqc

Index reference

In [ ]:
%%bash
bwa index ref/s_aureus.fna
samtools faidx ref/s_aureus.fna
ls ref

In [ ]:
%%bash
bwa mem -t 2 ref/s_aureus.fna \
  out/SRR33305440.trim.R1.fastq.gz \
  out/SRR33305440.trim.R2.fastq.gz \
| samtools sort -@ 2 -o map/SRR33305440.sorted.bam

samtools index map/SRR33305440.sorted.bam

ls map

Mapping

In [ ]:
%%bash
bwa mem -t 2 ref/s_aureus.fna \
  out/SRR33305440.trim.R1.fastq.gz \
  out/SRR33305440.trim.R2.fastq.gz \
| samtools sort -@ 2 -o map/SRR33305440.sorted.bam
samtools index map/SRR33305440.sorted.bam
ls map

Metrics

In [ ]:
%%bash
samtools flagstat map/SRR33305440.sorted.bam > map/SRR33305440.flagstat.txt
samtools idxstats map/SRR33305440.sorted.bam > map/SRR33305440.idxstats.txt
samtools coverage map/SRR33305440.sorted.bam > map/SRR33305440.coverage.txt
samtools depth -aa map/SRR33305440.sorted.bam > map/SRR33305440.depth.txt
ls map

SRR33305440.coverage.txt
SRR33305440.depth.txt
SRR33305440.flagstat.txt
SRR33305440.idxstats.txt
SRR33305440.sorted.bam
SRR33305440.sorted.bam.bai


Đọc kết quả

In [ ]:
%%bash
echo "===== FLAGSTAT ====="
cat map/SRR33305440.flagstat.txt
echo
echo "===== IDXSTATS ====="
column -t map/SRR33305440.idxstats.txt
echo
echo "===== COVERAGE ====="
column -t map/SRR33305440.coverage.txt
echo
echo "===== DEPTH (first 20 lines) ====="
head -n 20 map/SRR33305440.depth.txt

In [ ]:
from google.colab import files
files.download("map/SRR33305440.flagstat.txt")
files.download("map/SRR33305440.idxstats.txt")
files.download("map/SRR33305440.coverage.txt")
files.download("map/SRR33305440.depth.txt")